**Overview**  
This notebook describes the process of finding the best acrhitecture for NN, that predicts the system state using RDKit descriptors

First, we import all the necessary modules and define device for calculations - it is GPU card (*cuda:0*)

In [3]:
import pandas as pd
import numpy as np
import pickle
import torch
import torch.nn as nn
device = 'cuda:0'

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, confusion_matrix
import wandb

Then login into wandb

In [4]:
wandb.login(key = '32fdd85d94d1d691086ba1adae4171709b3cdc5d')

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: timyun. Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\Timur\.netrc


True

And load data drom two datasets - Standard and Padel

In [5]:
with open('X_standard_dropped.pickle', 'rb') as inp:
    X_standard = pickle.load(inp)

with open('X_padel_dropped.pickle', 'rb') as inp:
    X_padel = pickle.load(inp)

with open('y.pickle', 'rb') as inp:
    y = pickle.load(inp)

Let's create X and y dataframes both for PaDEL and Standard descriptors. After that, we scale them with MinMax scaler - it will save fp columns and change only numerical ones

In [6]:
X_train_standard, X_test_standard, y_train_standard, y_test_standard = train_test_split(X_standard, y, test_size= 0.2, random_state = 0)
X_valid_standard, X_test_standard, y_valid_standard, y_test_standard = train_test_split(X_test_standard, y_test_standard, test_size= 0.5, random_state = 0)
scaler = MinMaxScaler()
X_train_standard = scaler.fit_transform(X_train_standard)
X_test_standard = scaler.transform(X_test_standard)
X_valid_standard = scaler.transform(X_valid_standard)


In [7]:
X_train_padel, X_test_padel, y_train_padel, y_test_padel = train_test_split(X_padel, y, test_size= 0.2, random_state = 0)
X_valid_padel, X_test_padel, y_valid_padel, y_test_padel = train_test_split(X_test_padel, y_test_padel, test_size= 0.5, random_state = 0)

scaler = MinMaxScaler()
X_train_padel = scaler.fit_transform(X_train_padel)
X_test_padel = scaler.transform(X_test_padel)
X_valid_padel = scaler.transform(X_valid_padel)

Then we create torch datasets and dataloaders - *train*, *test*, *validation* sets for Standard and PaDEL descriptors

In [8]:
X_train_ds_stand = TensorDataset(torch.tensor(X_train_standard, dtype = torch.float32), torch.tensor(y_train_standard.to_numpy(), dtype = torch.long))
X_valid_ds_stand = TensorDataset(torch.tensor(X_valid_standard, dtype = torch.float32), torch.tensor(y_valid_standard.to_numpy(), dtype = torch.long))
X_test_ds_stand = TensorDataset(torch.tensor(X_test_standard, dtype = torch.float32), torch.tensor(y_test_standard.to_numpy(),dtype = torch.long))

X_train_stand_dl = DataLoader(X_train_ds_stand, batch_size = 12)
X_valid_stand_dl = DataLoader(X_valid_ds_stand, batch_size = 12)
X_test_stand_dl = DataLoader(X_test_ds_stand, batch_size = 12)



X_train_ds_padel = TensorDataset(torch.tensor(X_train_padel, dtype = torch.float32), torch.tensor(y_train_padel.to_numpy(), dtype = torch.long))
X_valid_ds_padel = TensorDataset(torch.tensor(X_valid_padel, dtype = torch.float32), torch.tensor(y_valid_padel.to_numpy(), dtype = torch.long))
X_test_ds_padel = TensorDataset(torch.tensor(X_test_padel, dtype = torch.float32), torch.tensor(y_test_padel.to_numpy(), dtype = torch.long))

X_train_padel_dl = DataLoader(X_train_ds_padel, batch_size = 12)
X_valid_padel_dl = DataLoader(X_valid_ds_padel, batch_size = 12)
X_test_padel_dl = DataLoader(X_test_ds_padel, batch_size = 12)

Then we just create our instance of classification Neural Network, which will take descriptors datase and  transofrms it into [1,2] tensor - logits for 0 class ('No gel') and 1 class ('Is gel'). First parameter of Neural Network is *n_descriptors*, which depends on the type of Dataset, i.e. Standard (1871) or Padel(1286)

We will change the NN it during search in Wandb. 

In [58]:
class Classification_NN(nn.Module):
    def __init__(self, n_descriptors = 1871, n1 = 2048, n2 = 2048, n3 = 2048):
        super().__init__()
        self.linear_1 = nn.Linear(n_descriptors, n1)
        self.a1 = nn.Mish()
        #self.dropout_1 = nn.Dropout(p = 0.3)
        self.linear_2 = nn.Linear(n1, n2)
        self.a2 = nn.Mish()

        self.linear_3 = nn.Linear(n2, n3)
        self.a3 = nn.Mish()
        
        self.linear_4 = nn.Linear(n3, 2)
    def forward(self, x):
        x = self.a1(self.linear_1(x))
        x = self.a2(self.linear_2(x))
        x = self.a3(self.linear_3(x))
        x = self.linear_4(x)
        return x

Let's check, that it works with one of our datasets 

In [52]:
model = Classification_NN()
model(X_test_ds_stand.tensors[0])

tensor([[0.0232, 0.0052],
        [0.0222, 0.0033],
        [0.0233, 0.0039],
        [0.0221, 0.0065],
        [0.0239, 0.0055],
        [0.0223, 0.0050],
        [0.0208, 0.0057],
        [0.0208, 0.0053],
        [0.0239, 0.0052],
        [0.0221, 0.0052],
        [0.0217, 0.0051],
        [0.0247, 0.0048],
        [0.0212, 0.0041],
        [0.0232, 0.0066],
        [0.0225, 0.0052],
        [0.0227, 0.0050],
        [0.0246, 0.0055],
        [0.0247, 0.0048],
        [0.0218, 0.0065],
        [0.0229, 0.0040],
        [0.0235, 0.0041],
        [0.0238, 0.0068],
        [0.0253, 0.0058],
        [0.0219, 0.0053],
        [0.0219, 0.0052],
        [0.0240, 0.0046],
        [0.0236, 0.0058],
        [0.0221, 0.0066],
        [0.0222, 0.0065],
        [0.0230, 0.0041],
        [0.0241, 0.0053],
        [0.0205, 0.0031],
        [0.0232, 0.0052],
        [0.0226, 0.0051],
        [0.0245, 0.0053],
        [0.0236, 0.0038],
        [0.0242, 0.0059],
        [0.0222, 0.0024],
        [0.0

One the next step we define:  
1. Model and the device for it
2. Number of epochs
3. Loss function
4. Learning rate
5. Optimizer with regularization feature ("weight decay")

And initialize *wandb* instance, which transfers the crucial information about properties and NN architecture into the project

In [59]:
model = Classification_NN()
model.to(device)
epochs = 350
loss_fn = nn.CrossEntropyLoss()
model.to(device)
lr = 0.001
optimizer = torch.optim.Adam(params = model.parameters(), lr = lr, weight_decay = 1e-4)
wandb.init(project = 'CTAB-hydrotropes',
           config = {'lr':lr,
                    'epochs':epochs,
                     'architecture': 'Linear (1871 -> 2048) -> Mish ->  -> Linear(2048 -> 2048) ->  Mish -> Linear(2048 -> 2048) -> Mish -> Linear(2048 -> 2)',
                     'manual_seed':'constant'
                    })
    
                      
           

train_accuracy,▁▆▅▆▆▆▆▇▆▇▆▆▇▇▇▇▇▆▇▇▆█▇▆▇▇▇▇▆▆▇▇█▇▇█▇▇▆▆
train_f1,▁▃▅▃▆▂▄▃▃▃▅▄▅▅▂▃▄▂▄▅▄▅▄▅▆▆▆▇▄▅▆▄▅█▇█▇▅▅▃
train_precision,▁▄▄▄▅▆▅▆▅▆▅▆▅▆▆▆▇▇▇█▇▆▄▃▃▂▃▃▂▄▄▄▄▅▃▄▄▃▄▃
train_recall,▆▆▆▆▇▄▅▆▅▅▅▆▆▅▅▅▅▅▁▅███▆▇█▇▇█▇▇▇▇▇▇▇▇▇▇▇
train_roc_auc_score,▆▁▁▁▃█▃▃▁▁▃▃▁▆▃▃▆▃▃▃▃▃▆▆█▆▃▃▆▁▁▁▁▁▁▁▁▁▁▁
valid_accuracy,▄█▄▆▅▆▅▆▅▅▇▆▆▂▄▆▄▄▅▆▆▅▂▁▂▇▆▅▅▇▇▄▇▇▇▅▇▆▆▇
valid_f1,▆▆▇▇▆▆▇▇▇▇▅▇▇▅█▇▅▅▃▆▅▁▆▂▅▇▇▇▇▆▅█▅▅▇█▇█▇▅
valid_precision,▅▆▇▄▆▆▆██▆▆▆█▇▆▆▇▇▆▇▇█▁▂▆▆▅▆▆▆▅▆▆▆▇▇▆▅▆▇
valid_recall,▆▄█▆▆▂▂▅▃▇▃▅▅▃▃▃▁▁▇▆▄▄▄▅▄▆▇▅▆▆▄▆▆▆▇▆▆▅▅▄
valid_roc_auc_score,▆▃▃▃▄▂▃▄▆▂▄▃▂▁▁▃▂▁▁▁▂▁██▆▇▆▆▂▇████▆█▆▇▆▃
train_accuracy,0.80285


In this step we assess the NN performance in the training and validation loop and transfer results into wandb

In [60]:
torch.manual_seed(0)
for epoch in range(epochs):
    #training loop
    model.train()
    train_epoch_accuracy, train_epoch_f1, train_epoch_precision, train_epoch_recall, train_epoch_roc_auc = 0, 0, 0, 0, 0
    for X_b, y_b in X_train_stand_dl:
        X_b = X_b.to(device)
        y_b = y_b.to(device)
        out = model(X_b)
        loss = loss_fn(out, y_b)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        out_labels = torch.argmax(out, dim = 1).cpu().detach().numpy()
        train_epoch_accuracy += accuracy_score(out_labels, y_b.cpu().detach().numpy())
        train_epoch_f1 += f1_score(y_b.cpu().detach().numpy(), out_labels)
        train_epoch_precision += precision_score(y_b.cpu().detach().numpy(), out_labels, zero_division=0)
        train_epoch_recall += recall_score(y_b.cpu().detach().numpy(), out_labels)
        train_epoch_roc_auc = roc_auc_score(y_b.cpu().detach().numpy(), out_labels)
    train_epoch_accuracy = train_epoch_accuracy/len(X_train_stand_dl)
    train_epoch_f1 = train_epoch_f1/len(X_train_stand_dl)
    train_epoch_precision = train_epoch_precision/len(X_train_stand_dl)
    train_epoch_recall = train_epoch_recall/len(X_train_stand_dl)
    train_epoch_roc_auc = train_epoch_roc_auc / len(X_train_stand_dl)

    wandb.log({'train_accuracy':train_epoch_accuracy,
               'train_f1':train_epoch_f1,
               'train_precision':train_epoch_precision,
               'train_recall':train_epoch_recall,
               'train_roc_auc_score':train_epoch_roc_auc})
    
    #validation loop
    valid_epoch_accuracy, valid_epoch_f1, valid_epoch_precision, valid_epoch_recall, valid_epoch_roc_auc = 0, 0, 0, 0, 0
    model.eval()
    for X_b, y_b in X_valid_stand_dl:
        X_b = X_b.to(device)
        y_b = y_b.to(device)
        out = model(X_b)
        loss = loss_fn(out, y_b)
        out_labels = torch.argmax(out, dim = 1).cpu().detach().numpy()
        valid_epoch_accuracy += accuracy_score(out_labels, y_b.cpu().detach().numpy())
        valid_epoch_f1 += f1_score(y_b.cpu().detach().numpy(), out_labels)
        valid_epoch_precision += precision_score(y_b.cpu().detach().numpy(), out_labels, zero_division=0)
        valid_epoch_recall += recall_score(y_b.cpu().detach().numpy(), out_labels)
        valid_epoch_roc_auc = roc_auc_score(y_b.cpu().detach().numpy(), out_labels)
    valid_epoch_accuracy /= len(X_valid_stand_dl)
    valid_epoch_f1 /= len(X_valid_stand_dl)
    valid_epoch_precision /= len(X_valid_stand_dl)
    valid_epoch_recall /= len(X_valid_stand_dl)
    valid_epoch_roc_auc /= len(X_valid_stand_dl)
    wandb.log({'valid_accuracy':valid_epoch_accuracy,
               'valid_f1':valid_epoch_f1,
               'valid_precision':valid_epoch_precision,
               'valid_recall':valid_epoch_recall,
               'valid_roc_auc_score':valid_epoch_roc_auc})
    if epoch % 10 ==0:
        print('Valid accuracy on is {}, f1 is {}, precision is {}, recall is {}, ROC-AUC is {} -- epochs{}'.format(valid_epoch_accuracy,
                                                                                                     valid_epoch_f1, 
                                                                                                     valid_epoch_precision,
                                                                                                     valid_epoch_recall, 
                                                                                                     valid_epoch_roc_auc,
                                                                                                                   epoch))

Valid accuracy on is 0.5166666666666667, f1 is 0.6759958720330237, precision is 0.5166666666666667, recall is 1.0, ROC-AUC is 0.1 -- epochs0
Valid accuracy on is 0.6333333333333333, f1 is 0.6777777777777778, precision is 0.6155555555555555, recall is 0.7723809523809525, ROC-AUC is 0.15 -- epochs10
Valid accuracy on is 0.6833333333333333, f1 is 0.7043858102681633, precision is 0.6663492063492064, recall is 0.7676190476190475, ROC-AUC is 0.175 -- epochs20
Valid accuracy on is 0.6833333333333333, f1 is 0.6696338955162485, precision is 0.6877777777777777, recall is 0.6657142857142857, ROC-AUC is 0.175 -- epochs30
Valid accuracy on is 0.6166666666666666, f1 is 0.5205128205128206, precision is 0.75, recall is 0.4307142857142857, ROC-AUC is 0.1625 -- epochs40
Valid accuracy on is 0.6666666666666667, f1 is 0.6523232323232323, precision is 0.6599999999999999, recall is 0.659047619047619, ROC-AUC is 0.15 -- epochs50
Valid accuracy on is 0.6666666666666667, f1 is 0.6956526806526806, precision is 

This step is made to calculate accuracy of the model on the test dataset, just to check consistency of results, obtained duting evaluation loop on the validation dataset with results for test dataset

In [19]:
accuracy_score(torch.argmax(model(X_test_ds_stand.tensors[0].to(device)), dim = 1).cpu().detach().numpy(), y_test_standard)

0.7377049180327869